<a href="https://colab.research.google.com/github/Tin-Tin-04/Gen_ai_2025/blob/main/GenAI_Project_AyushMajumdar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch gradio --quiet


In [ ]:
from transformers import pipeline

bart_summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

input_text = """Artificial Intelligence is transforming industries by automating tasks,
improving decision making, and enhancing human productivity. However, ethical
concerns and the need for regulation continue to grow alongside its rapid development."""

summary_result = bart_summarizer(input_text, max_length=60, min_length=20, do_sample=False)

print("Generated Summary:\n", summary_result[0]['summary_text'])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu
Your max_length is set to 60, but your input_length is only 43. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=21)


Summary:
 Artificial Intelligence is transforming industries by automating tasks,improving decision making, and enhancing human productivity. But ethical concerns and the need for regulation continue to grow alongside its rapid development.


In [ ]:
import re
from collections import Counter

def calculate_compression(original_text, summary_text):
    original_word_count = len(original_text.split())
    summary_word_count = len(summary_text.split())
    compression_percentage = (summary_word_count / original_word_count) * 100
    return original_word_count, summary_word_count, compression_percentage

def extract_summary_keywords(text, top_n=5):
    words = re.findall(r'\b\w+\b', text.lower())
    stopwords = {"the","is","and","a","an","of","to","in","that","it","for","as","on","with","this","by","are","was"}
    meaningful_words = [word for word in words if word not in stopwords]
    word_frequencies = Counter(meaningful_words)
    return [word for word, count in word_frequencies.most_common(top_n)]

def display_summary_analysis(original_text, summary_text):
    original_count, summary_count, compression = calculate_compression(original_text, summary_text)
    keywords = extract_summary_keywords(summary_text)

    print("\n--- Extra Analysis ---")
    print(f"Original Word Count: {original_count}")
    print(f"Summary Word Count: {summary_count}")
    print(f"Compression Ratio: {compression:.2f}%")
    print(f"Top Keywords from Summary: {keywords}")


In [ ]:
import gradio as gr
from transformers import pipeline

bart_model = pipeline("summarization", model="facebook/bart-large-cnn")
t5_model = pipeline("summarization", model="t5-small")

def get_compression_stats(original_text, summary_text):
    original_words = len(original_text.split())
    summary_words = len(summary_text.split())
    compression = (summary_words / original_words) * 100
    return original_words, summary_words, compression

def get_top_keywords(text, top_n=5):
    words = text.lower().split()
    frequency = {}
    for word in words:
        if word.isalpha():
            frequency[word] = frequency.get(word, 0) + 1
    sorted_words = sorted(frequency.items(), key=lambda x: x[1], reverse=True)
    return [word for word, count in sorted_words[:top_n]]

def generate_summary(text, selected_model, length_preference):
    if not text.strip():
        return "Please enter some text to summarize."

    if length_preference == "Short":
        max_len, min_len = 60, 20
    elif length_preference == "Medium":
        max_len, min_len = 100, 40
    else:
        max_len, min_len = 140, 60

    if selected_model == "BART (facebook/bart-large-cnn)":
        summary_text = bart_model(text, max_length=max_len, min_length=min_len, do_sample=False)[0]['summary_text']
    else:
        summary_text = t5_model("summarize: " + text, max_length=max_len, min_length=min_len, do_sample=False)[0]['summary_text']

    original_count, summary_count, compression_ratio = get_compression_stats(text, summary_text)
    keywords = get_top_keywords(summary_text)

    analysis = (
        f"\n\n--- Extra Analysis ---\n"
        f"Original Word Count: {original_count}\n"
        f"Summary Word Count: {summary_count}\n"
        f"Compression Ratio: {compression_ratio:.2f}%\n"
        f"Top Keywords: {keywords}"
    )

    return summary_text + analysis

def compare_two_models(text, length_preference):
    if not text.strip():
        return "Enter text", "Enter text"

    if length_preference == "Short":
        max_len, min_len = 60, 20
    elif length_preference == "Medium":
        max_len, min_len = 100, 40
    else:
        max_len, min_len = 140, 60

    t5_summary = t5_model("summarize: " + text, max_length=max_len, min_length=min_len, do_sample=False)[0]["summary_text"]
    bart_summary = bart_model(text, max_length=max_len, min_length=min_len, do_sample=False)[0]["summary_text"]

    return t5_summary, bart_summary

interface = gr.Interface(
    fn=generate_summary,
    inputs=[
        gr.Textbox(lines=8, placeholder="Enter text to summarize here...", label="Input Text"),
        gr.Radio(["BART (facebook/bart-large-cnn)", "T5 (t5-small)"], label="Choose Model"),
        gr.Radio(["Short", "Medium", "Long"], label="Summary Length"),
    ],
    outputs=gr.Textbox(label="Generated Summary"),
    title="Generative AI Text Summarizer",
    description="Compare BART and T5 transformer models for automatic text summarization. Choose the summary length and model type to see differences.",
    allow_flagging="never",
)

interface.launch(share=True)


Device set to use cpu
Device set to use cpu
/usr/local/lib/python3.12/dist-packages/gradio/interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f91a0eaced02d8da32.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr

model_comparison_interface = gr.Interface(
    fn=compare_two_models,
    inputs=[
        gr.Textbox(lines=8, placeholder="Enter text to summarize here...", label="Input Text"),
        gr.Radio(["Short", "Medium", "Long"], label="Summary Length"),
    ],
    outputs=[
        gr.Textbox(label="T5 Summary"),
        gr.Textbox(label="BART Summary"),
    ],
    title="Model Comparison: T5 vs BART",
    description="Generate summaries from both models and compare them side by side."
)

model_comparison_interface.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fbc66638da543abf97.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import re
from collections import Counter

def get_compression_stats(original_text, summary_text):
    original_words = len(original_text.split())
    summary_words = len(summary_text.split())
    compression_ratio = (summary_words / original_words) * 100
    return original_words, summary_words, compression_ratio

def get_top_keywords(text, top_n=5):
    words = re.findall(r'\b\w+\b', text.lower())
    stopwords = {"the","is","and","a","an","of","to","in","that","it","for","as","on","with","this","by","are","was"}
    filtered_words = [word for word in words if word not in stopwords]
    word_counts = Counter(filtered_words)
    return [word for word, count in word_counts.most_common(top_n)]

def show_summary_analysis(original_text, summary_text):
    original_count, summary_count, compression = get_compression_stats(original_text, summary_text)
    keywords = get_top_keywords(summary_text)

    print("\n--- Extra Analysis ---")
    print(f"Original Word Count: {original_count}")
    print(f"Summary Word Count: {summary_count}")
    print(f"Compression Ratio: {compression:.2f}%")
    print(f"Top Keywords from Summary: {keywords}")
